# 03 — Modeling: baseline vs ensemble

## Obiettivi didattici

1. Confrontare **due famiglie** di modelli: lineare regolarizzato (Logistic Regression) e ensemble di alberi (Random Forest).
2. Applicare **time-series cross-validation** (no shuffle).
3. Gestire lo sbilanciamento via **`class_weight='balanced'`**.
4. (Opzionale) Aggiungere XGBoost se installato.
5. Selezionare il miglior modello via **AUC-PR su CV**.
!!! note "Dataset richiesto"
    Il dataset Kaggle (~470MB) NON e' in repo per limiti di GitHub.
    Scaricalo da <https://www.kaggle.com/datasets/kartik2112/fraud-detection>
    e copia `fraudTrain.csv` e `fraudTest.csv` in `data/raw/`.


In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from fraud_pipeline.data import load_train_test, split_features_target, downsample_for_smoke_test
from fraud_pipeline.features import FraudFeatureEngineer
from fraud_pipeline.preprocessing import build_preprocessor, infer_column_groups
from fraud_pipeline.models import get_all_pipelines
from fraud_pipeline.tuning import tune_all_models, summarize_tuning, PRIMARY_SCORING
from fraud_pipeline.config import DEFAULT_CONFIG
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score


## Setup dati: sample 100k per smoke test

In [ ]:
df_train, df_test = load_train_test()
df_train = downsample_for_smoke_test(df_train, 100_000,
                                     random_state=DEFAULT_CONFIG.random_state)
X_train, y_train = split_features_target(df_train)
X_test, y_test = split_features_target(df_test)

fe = FraudFeatureEngineer()
X_train_fe = fe.fit_transform(X_train)
groups = infer_column_groups(X_train_fe)
preproc = build_preprocessor(numeric_cols=groups['numeric'],
                             nominal_cols=groups['nominal'])

n_pos = max(int(y_train.sum()), 1)
scale_pos_weight = (len(y_train) - n_pos) / n_pos
print(f'scale_pos_weight = {scale_pos_weight:.1f}')

base = get_all_pipelines(preproc, use_class_weight=True,
                         include_xgboost=False,  # cambia a True se hai installato xgboost
                         scale_pos_weight=scale_pos_weight)
candidates = {n: Pipeline([('feature_engineer', FraudFeatureEngineer())] + list(p.steps))
              for n, p in base.items()}
list(candidates.keys())


## Baseline cross-validation (no tuning)

Misura l'AUC-PR dei modelli con iperparametri di default su **TimeSeriesSplit** (4 fold). Niente shuffle: i fold rispettano l'ordine temporale.

In [ ]:
cv = TimeSeriesSplit(n_splits=DEFAULT_CONFIG.cv_splits)
rows = []
for name, pipe in candidates.items():
    scores = cross_val_score(pipe, X_train, y_train,
                             scoring=PRIMARY_SCORING, cv=cv, n_jobs=-1)
    rows.append({'model': name, 'auc_pr_mean': scores.mean(), 'auc_pr_std': scores.std()})
pd.DataFrame(rows).sort_values('auc_pr_mean', ascending=False)


## Tuning iperparametri

**Strategia per modello**:

- **LogisticRegression**: GridSearchCV su `C` (forza regolarizzazione).
- **RandomForest**: GridSearchCV su `n_estimators`, `max_depth`, `min_samples_leaf`.
- **XGBoost** (se incluso): RandomizedSearchCV su grid combinatoria.

**Scoring primario**: `average_precision` (= AUC-PR).


In [ ]:
results = tune_all_models(
    pipelines=candidates,
    X=X_train, y=y_train,
    config=DEFAULT_CONFIG,
    xgb_n_iter=8,
)
summary = summarize_tuning(results)
summary


## Discussione

Tipicamente:

- **Random Forest** vince su Logistic Regression per AUC-PR di alcuni punti percentuali: cattura interazioni non lineari (es. importo alto + orario notturno + distanza grande).
- **Logistic Regression** rimane comunque utile come **modello interpretabile** per il business: i coefficienti sono leggibili come log-odds ratio.
- **XGBoost** (se incluso) tipicamente sorpassa RF di 1-3 punti su AUC-PR ma e' piu' costoso da tunare.


## Salvataggio modelli per il notebook successivo

In [ ]:
import joblib
from pathlib import Path
out_dir = Path('../reports/models')
out_dir.mkdir(parents=True, exist_ok=True)
for name, r in results.items():
    path = out_dir / f'{name.lower()}_best.joblib'
    joblib.dump(r.best_estimator, path)
    print(f'salvato: {path.name}  AUC-PR_cv={r.best_score:.4f}')
